## ПРАКТИЧЕСКАЯ РАБОТА №2.
«ИССЛЕДОВАНИЕ ДАННЫХ НА PYTHON. ОБРАБОТКА ВЫБРОСОВ.
ВИЗУАЛЬНЫЙ EDA»

In [123]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import warnings


In [124]:
def get_real_value(nom, old, new_cpi):
    return (nom * new_cpi) / old

## Загрузка данных


In [125]:
data = pd.read_csv("Dataset1.csv")
cpi_table = pd.read_csv("cpi.csv")
cpi_2016 = float(cpi_table[cpi_table["year"]==2016]["avg_cpi"].values[0])

## Предобработка данных

In [126]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 5043 entries, 0 to 5042
Data columns (total 28 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   color                      5025 non-null   str    
 1   Director_Name              4872 non-null   str    
 2   num_Critic_for_reviews     4927 non-null   float64
 3   duration                   4959 non-null   float64
 4   director_Facebook_likes    4872 non-null   float64
 5   actor_3_Facebook_likes     4953 non-null   float64
 6   actor_2_name               4963 non-null   str    
 7   Actor_1_Facebook_likes     4968 non-null   float64
 8   gross                      4104 non-null   float64
 9   genres                     4974 non-null   str    
 10  actor_1_name               4968 non-null   str    
 11  movie_Title                4974 non-null   str    
 12  num_voted_users            4974 non-null   float64
 13  cast_total_facebook_likes  4974 non-null   float64
 14  act

### Мы видим что датасет:
- состоит из 28 столбцов
- в нем 5043 строки
- данные представленны string и float

In [127]:
data.head(5)

,color,Director_Name,num_Critic_for_reviews,duration,director_Facebook_likes,actor_3_Facebook_likes,actor_2_name,Actor_1_Facebook_likes,gross,genres,...,num_user_for_reviews,language,country,content_rating,budget,title_year,actor_2_facebook_likes,imdb_score,aspect_ratio,movie_facebook_likes;
0,Color,James Cameron,723.0,178.0,0.0,855.0,Joel David Moore,1000.0,760505847.0,Action|Adventure|Fantasy|Sci-Fi,...,3054.0,English,USA,PG-13,237000000.0,2009.0,936.0,7.9,1.78,33000;
1,Colour,Gore Verbinski,302.0,169.0,563.0,1000.0,Orlando Bloom,40000.0,309404152.0,Action|Adventure|Fantasy,...,1238.0,English,USA,PG-13,300000000.0,2007.0,5000.0,7.1,2.35,0;
2,Colour,Sam Mendes,602.0,148.0,0.0,161.0,Rory Kinnear,11000.0,200074175.0,Action|Adventure|Thriller,...,994.0,English,UK,PG-13,245000000.0,2015.0,393.0,6.8,2.35,85000;
3,Color,Christopher Nolan,813.0,164.0,22000.0,23000.0,Christian Bale,27000.0,448130642.0,Action|Thriller,...,2701.0,English,USA,PG-13,250000000.0,2012.0,23000.0,8.5,2.35,164000;
4,NaN,Doug Walker,NaN,NaN,131.0,NaN,Rob Walker,131.0,NaN,Documentary,...,NaN,NaN,NaN,NaN,NaN,NaN,12.0,7.1,NaN,0;


## План на очистку данных 
- убрать ошибку представения датасета, когда все данные были записанны в одну ячейку
- найти и удалить дубликаты
- устранить пропуски

#### Удаление полностью пустых строк

In [128]:
old_data_size = data.shape[0]
data = data.drop(data[data.isna().sum(axis=1) == 27].index).reset_index(drop=True)
deleted = old_data_size - data.shape[0]
deleted

69

Мы избавились от 69 полность пустых строк 

#### Теперь переходим к удалению дубликатов

In [129]:
data.duplicated().sum()

np.int64(44)

In [130]:
data = data.drop_duplicates(ignore_index=True)

Мы удалили полные дубликаты данных

In [131]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 4930 entries, 0 to 4929
Data columns (total 28 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   color                      4912 non-null   str    
 1   Director_Name              4829 non-null   str    
 2   num_Critic_for_reviews     4884 non-null   float64
 3   duration                   4915 non-null   float64
 4   director_Facebook_likes    4829 non-null   float64
 5   actor_3_Facebook_likes     4909 non-null   float64
 6   actor_2_name               4919 non-null   str    
 7   Actor_1_Facebook_likes     4924 non-null   float64
 8   gross                      4070 non-null   float64
 9   genres                     4930 non-null   str    
 10  actor_1_name               4924 non-null   str    
 11  movie_Title                4930 non-null   str    
 12  num_voted_users            4930 non-null   float64
 13  cast_total_facebook_likes  4930 non-null   float64
 14  act

In [132]:
data.duplicated(subset="movie_Title").sum()

np.int64(80)

#### Здесь мы видим: 
- у нас есть строки которые описывают фильмы, которые уже есть в датасете
- в датасете отображены данные из одного источника и они не должны повторяться тк каждому фильму может быть выставленна только одна оценка 

Я решил что такие дубликаты скорее всего отлицаются в какой-то одной переменной на величину погрешности 

In [133]:
data = data.drop_duplicates(subset="movie_Title",ignore_index=True)

In [134]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 4850 entries, 0 to 4849
Data columns (total 28 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   color                      4832 non-null   str    
 1   Director_Name              4750 non-null   str    
 2   num_Critic_for_reviews     4804 non-null   float64
 3   duration                   4835 non-null   float64
 4   director_Facebook_likes    4750 non-null   float64
 5   actor_3_Facebook_likes     4829 non-null   float64
 6   actor_2_name               4839 non-null   str    
 7   Actor_1_Facebook_likes     4844 non-null   float64
 8   gross                      4000 non-null   float64
 9   genres                     4850 non-null   str    
 10  actor_1_name               4844 non-null   str    
 11  movie_Title                4850 non-null   str    
 12  num_voted_users            4850 non-null   float64
 13  cast_total_facebook_likes  4850 non-null   float64
 14  act

Мы разобрались со всеми возможными дубликатами

#### Приступаем к обработке пропусков

In [135]:
(data.isna().sum()/data.shape[0]*100).sort_values(ascending=False)

gross                        17.525773
budget                        9.752577
aspect_ratio                  6.536082
content_rating                6.061856
plot_keywords                 3.010309
title_year                    2.123711
Director_Name                 2.061856
director_Facebook_likes       2.061856
num_Critic_for_reviews        0.948454
actor_3_name                  0.432990
actor_3_Facebook_likes        0.432990
num_user_for_reviews          0.371134
color                         0.371134
duration                      0.309278
facenumber_in_poster          0.268041
language                      0.268041
actor_2_name                  0.226804
actor_2_facebook_likes        0.226804
actor_1_name                  0.123711
Actor_1_Facebook_likes        0.123711
country                       0.061856
cast_total_facebook_likes     0.000000
num_voted_users               0.000000
movie_Title                   0.000000
movie_imdb_link               0.000000
genres                   

##### Сначала разберемся с годом выапуска фильма, поскольку это признак нам поможет при заполнении других данных боллее точно на основании года их производства

In [136]:
df = data[data["title_year"].isna()]
df['movie_Title'].head(30)

4       Star Wars: Episode VII - The Force Awakens    ...
175                               Miami Vice             
255                               The A-Team             
394                                 Hannibal             
449                                Daredevil             
469                                Bewitched             
527                              Constantine             
533                                     Life             
582                            Sleepy Hollow             
634                        Last Man Standing             
673                              The Missing             
744                      Rules of Engagement             
811                         Sex and the City             
818                         Anger Management             
842                            Stargate SG-1             
944                              Unforgotten             
976                         A Touch of Frost             
989           

У нас есть целых 103 фильма без года его производства, и в их названии нет указания года которое могло бы нам помочь.
- мы можем попробовать заполнить год выпуска, по медиане для конкретного режиссера

In [137]:
data[data["title_year"].isna() & data["Director_Name"].isna()].shape[0]


100

##### Мы видим что почти во всех строках где пропущен режисер, пропущен и год

In [138]:
# data['title_year'].fillna(data['title_year'].median(), inplace=True)

## План на преобразование данных

- преобразовать float стобцы в которых содержатся только целочисленные данные в int